In [ ]:
import pandas as pd
import numpy as np
import re
import json
import scipy

# Function to parse individual entries (e.g., A0:0:7:86)
def parse_entry(entry):
    parts = entry.split(":")
    # Clean and convert values
    clean = lambda x: float(x.replace("‑", "-").replace("−", "-"))  # Replace non-standard hyphens
    if len(parts) == 4:
        key, start, length, value = parts
        return {"Encoder1": key[1:], "Encoder2": clean(start), "Voltage": clean(length), "phi": clean(value)}
    elif len(parts) == 3:
        phi, phi_dot, servo = parts
        return {"phi":phi[1:], "phi_dot": clean(phi_dot), "Servo": clean(servo)}
    else:
        return {"raw_entry": entry}  # Fallback for unexpected formats

In [ ]:
# Specify the file name
# file_name = "data-limited2"  # Replace with your file name
# file_name = "Good_no_fallings_original_data:large_left_circle"
# file_name = "bike_in_air_data"
# file_name = "bike_pushed_data"
# file_name = "bike_pushed_data2"
# file_name = "medium_left_circle_with_float_data"
# file_name = "medium_left_circle_with_float_data+noisy_input"

file_names = []
# file_names.append("data-limited2")
# file_names.append("Good_no_fallings_original_data:large_left_circle")
# file_names.append("bike_in_air_data")
# file_names.append("bike_pushed_data")
# file_names.append("bike_pushed_data2")
# file_names.append("medium_left_circle_with_float_data")
file_names.append("medium_left_circle_with_float_data+noisy_input")
data = ""

for file_name in file_names:
    print("Processing file: ", file_name)
    # Read the content of the file
    with open(file_name, "r") as file:
        data += file.read()

# Pattern to match ${A..} and ${B..} blocks
# pattern = r"\$\{([AB]\d+:[^}]+)\}"
pattern = r"\$\{([AB]\d+\.\d+:[^}]+)\}"

# Extract matches using regex
matches = re.findall(pattern, data)

# Process and categorize the matches
data_blocks = {"A": [], "B": []}
for match in matches:
    if match.startswith("A"):
        data_blocks["A"].append(match)
    elif match.startswith("B"):
        data_blocks["B"].append(match)



# Parse data for A and B blocks
parsed_data = {"A": [parse_entry(entry) for entry in data_blocks["A"]],
               "B": [parse_entry(entry) for entry in data_blocks["B"]]}
# Example: Output the processed data
print("Length of data B:", len(parsed_data["B"]))

print("Parsed A data:")
# for item in parsed_data["A"]:
#     print(item)

print("\nParsed B data:")
# for item in parsed_data["B"]:
#     print(item)

# Optionally, save to a file
with open("parsed_data.json", "w") as f:
    import json
    json.dump(parsed_data, f, indent=4)

# Further processing can be added as needed


In [ ]:
# Load the JSON file
json_file = "parsed_data.json"  # Ensure this is the correct path to your JSON file

with open(json_file, "r") as f:
    data = json.load(f)

# Convert the "A" and "B" blocks to pandas DataFrames
df_a = pd.DataFrame(data["A"], columns=["Encoder1", "Encoder2", "Voltage", "phi"], dtype=float)
df_b = pd.DataFrame(data["B"], columns=["phi", "phi_dot", "Servo"], dtype=float)

# Display the DataFrames
print("DataFrame for A Block:")
print(df_a)

print("\nDataFrame for B Block:")
print(df_b)

# Save as CSV for further analysis if needed
df_a.to_csv("parsed_A_data.csv", index=False)
df_b.to_csv("parsed_B_data.csv", index=False)


## Don't normalise, but fix to radians
MID = 750
delta = (df_b['Servo'] - MID) / 3.65
df_b['delta'] = delta


df_b['delta'] = df_b['delta']
# df_b['phi_rad'] = df_b['phi'] / 180 * np.pi
# df_b['phi_dot'] = df_b['phi_dot'] / 180 * np.pi

In [ ]:
# delta.plot.hist(bins=100) #   -35 < delta < 36    -22.5 < delta < -5 [[when circling to the left]]
# (df_b['delta']).plot.hist(bins=100)
# df_b['phi_dot'].plot.hist(bins=50)
# df_b['phi_dot'].unique
df_b['phi'].plot.hist(bins=50)
# df_b['delta']

In [ ]:
clean_states = df_b[df_b['phi'].abs() <= 45]
# clean_states = df_b
print(f'length(clean_states): {len(clean_states)}')
clean_states = clean_states.drop(columns=['Servo'])[['phi', 'phi_dot', 'delta']]

# Add noise to phi
noise = np.random.normal(0, 3, clean_states.shape[0])
noise = 0
clean_states['phi'] = clean_states['phi'] + noise

# noise = np.random.normal(0, 0.1, clean_states.shape[0])
# clean_states['phi_dot'] = clean_states['phi_dot'] + noise

# noise = np.random.normal(0, 0.1, clean_states.shape[0])
# clean_states['delta'] = clean_states['delta'] + noise

# clean_states['phi_dot'].plot.hist(bins=50)
# clean_states['phi_dot'].head()
# clean_states['delta'].head()
# noise

In [ ]:
# # Compute phi dot (OPTIONAL: It is already provided in the data)
Ts = 0.020
# clean_states['phi_dot'] = clean_states['phi'].diff()
# clean_states['phi_dot'] = clean_states['phi_dot'].fillna(0)
# clean_states['phi_dot'] = clean_states['phi_dot'].shift(-1)
# clean_states['phi_dot'] = clean_states['phi_dot'].fillna(0) / Ts
# clean_states['phi_dot'].unique()
# clean_states['phi'].unique()

In [ ]:
## Compute delta dot for matlab_inputs_u_g
Ts = 0.020
clean_states['delta_dot'] = clean_states['delta'].diff()
clean_states['delta_dot'] = clean_states['delta_dot'].fillna(0)
clean_states['delta_dot'] = clean_states['delta_dot'].shift(-1)
clean_states['delta_dot'] = clean_states['delta_dot'].fillna(0) / Ts

In [ ]:
clean_states['phi'].plot.line()

In [ ]:
# Study epsilon with real system matrices
from scipy.signal import cont2discrete, lti, dlti, dstep
epsilon = 0

# Bike data
g = 9.8
h = 0.088 # m (hieght of the centre of mass)
v = 0.634 # m/s or v = 0.634
a = 0.055 # m (distance between rear wheel and centre of gravity projection)
w = 0.167 # m (Distance between front and rear wheels and ground contact points)
lambd =  75/180 * np.pi # in rad = 70 degrees  (fork angle)
# lambd =  90/180 * np.pi # in rad = 70 degrees  (fork angle)
wheel_radius = 0.0375 # m (diameter is around 7.5cm)
r_tau = wheel_radius * np.tan(np.pi/2 - lambd) # (distance between front wheel and intersection of fork with ground)
l = w; # length

A23 = -(v**2*h-a*r_tau*g) * np.sin(lambd) / (h**2*l);
B2 = - a*v*np.sin(lambd)/(h*l) # Try with negative: yes

A_c = np.array([[0, 1, 0],
                  [g/h, 0, A23],
                  [0, 0, 0]])
B_c = np.array([[0],
                [B2],
                [1]])
sys = cont2discrete((A_c, B_c, np.array([[1, 0, 0]]), [0]), dt=0.02, method='zoh')
A_s = sys[0]
B_s = sys[1]

In [ ]:
# A_s = np.array([[1.0224, 0.0201, -0.0045],
#                 [2.2438, 1.0224, -0.4508],
#                 [0, 0, 1]])
# B_s = np.array([[-0.0005],
#                 [-0.0507],
#                 [0.0200]])

states_only = clean_states.drop(['delta_dot'], axis=1)
count_bad_data = 0
count_sequence_with_good_data = 0
longest_sequence_with_good_data = 0
indices = []
good_indices = []
perfect_indices = []
next_perfect_states_indices = []
epsilon = 0

arg_max = [0,0,0]

for i in range(len(states_only)):
    if i == 0:
        continue
    indices.append(i)
    count_sequence_with_good_data += 1

    current_state = states_only.iloc[i].to_numpy()[np.newaxis].transpose()
    previous_state = states_only.iloc[i-1]
    # print(i)
    predict = (A_s @ previous_state)[np.newaxis].transpose() + B_s * clean_states['delta_dot'].iloc[i-1]
    # predict[1] = np.floor(predict[1])

    current_state = current_state * np.pi / 180
    predict = predict * np.pi / 180
    # current_state = current_state * np.pi / 180
    # predict = predict * np.pi / 180

    # current_state[0] = current_state[0] * np.pi/180
    # predict[0] = predict[0] * np.pi/180
    # # current_state[1] = current_state[1] * np.pi/180
    # predict[1] = np.floor(predict[1]) * np.pi/180
    # current_state[1] = predict[1]
    # #####################################################
    # # clean_states['phi_dot'][i] = predict[1]
    # #####################################################
    # current_state[2] = current_state[2] * np.pi/180
    # predict[2] = predict[2] * np.pi/180



    error = current_state - predict

    # print("error:", error)
    # print("argmax error:", error.argmax())
    arg_max[error.argmax()] += 1

    epsilon = max(np.linalg.norm(error, 2), epsilon)
    # if np.linalg.norm(error, 2) > 1:
    #     print(error)

    ## Check if the data has no longer good errors
    if np.linalg.norm(error, 2) > 10e-1:
        count_bad_data += 1
        if count_sequence_with_good_data > longest_sequence_with_good_data:
            good_indices = indices
        indices = []
        longest_sequence_with_good_data = max(longest_sequence_with_good_data, count_sequence_with_good_data)
        count_sequence_with_good_data = 0
        # print(np.linalg.norm(error, 2))
        # print(i)
        # print("predict:", predict)
        # print("current_state:", current_state)

    ## Check if the data has "perfect" errors
    if np.linalg.norm(error, 2) < 0.001:
        perfect_indices.append(i)
        # print(current_state*180/np.pi)


next_perfect_states_indices = [i+1 for i in perfect_indices]
if next_perfect_states_indices[-1] == len(states_only):
    del perfect_indices[-1]
    del next_perfect_states_indices[-1]

print("count_bad_data", count_bad_data)
print("longest_sequence_with_good_data", longest_sequence_with_good_data)
print("good_indices", good_indices, '\n')
print("count perfect data", len(perfect_indices))
print("perfect_indices", perfect_indices, '\n')

print("epsilon = ", epsilon)

arg_max

In [ ]:
# error



In [ ]:
# states_only.iloc[1].to_numpy()[np.newaxis].transpose()
# states_only['phi_dot'].unique()

In [ ]:
len(states_only)
perfect_indices[-1] == len(states_only)

In [ ]:
print(states_only.max())
print()
print(states_only.min())


## Save Matlab Data

In [ ]:
# clean_states.head()
# clean_states = clean_states[100:116]
# clean_states = clean_states[339:442]
# clean_states = clean_states.iloc[perfect_indices]


In [ ]:
# save_data = clean_states.iloc[perfect_indices] * np.pi / 180
save_data = clean_states.iloc[perfect_indices]
save_next_states = clean_states.iloc[next_perfect_states_indices]

# save_data = clean_states
matlab_states_x_g = pd.Series([val for pair in zip(save_data['phi'], save_data['phi_dot'], save_data['delta']) for val in pair])
# matlab_states_x_g

matlab_next_states = pd.Series([val for pair in zip(save_next_states['phi'], save_next_states['phi_dot'], save_next_states['delta']) for val in pair])

In [ ]:
# matlab_inputs_u_g = clean_states['delta_dot'] * np.pi / 180
matlab_inputs_u_g = clean_states['delta_dot']
# matlab_inputs_u_g = clean_states['delta']

In [ ]:

# matlab_data = {'x_g': matlab_states_x_g, 'u_g': matlab_inputs_u_g, 'next_states':matlab_next_states}
matlab_data = {'x_g': matlab_states_x_g * np.pi / 180, 'u_g': matlab_inputs_u_g * np.pi / 180, 'next_states':matlab_next_states * np.pi / 180} # Save in radians to decrease the epsilon (error)
scipy.io.savemat('matlab_data.mat', matlab_data, oned_as='column')